# Lab 4: Random Forests (RF)

*FW 532 · Python version*

**Goals:**

* To understand how random forest models are built and interpreted in Python.

**You Will Need:**

* The course environmental raster stack (`data/envStack.tif`, included in the starter kit)
* Python packages: numpy, pandas, matplotlib, scikit-learn, rasterio (see `requirements.txt`)
* The course helper module `sdmlab.py` (in the same folder as this notebook)

Run the cells in order from top to bottom: later cells use variables created by earlier ones. A few
cells (the 5000-tree forests and the prediction map) take up to a minute.

In [ ]:
# Running on Google Colab? This cell downloads the course files. (Anywhere else it does nothing.)
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/fw532"):
        !git clone -q --depth 1 https://github.com/Hutchinson-Lab/fw532.git /content/fw532
        %pip install -q rasterio
    os.chdir("/content/fw532/python")

# Setup

New this week: scikit-learn's random forests, `RandomForestClassifier` and `RandomForestRegressor`.
They play the role of R's randomForest and ranger packages. We'll also use two of scikit-learn's
tools for interpreting models, `permutation_importance` and `PartialDependenceDisplay`.

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

import sdmlab as sdm   # course helpers for rasters

env_stack = sdm.load_env()

# Generate Data

This is the same data simulation as last lab, with a nonlinear function of the environmental
variables. Feel free to experiment as well with the original linear version (commented out here) or
a function of your own choosing.

(Python and R draw different random numbers, so your points, and every number below, will differ a
little from the R version of this lab.)

In [ ]:
# define logistic function
def logistic(x):
    return np.exp(x) / (1 + np.exp(x))

# get points
n_points = 1000
sim_pts = env_stack.sample_random(n_points, seed=1234)

# set parameters
# intercept = -1.5
# beta1 = -2
# beta2 = -1.5
# beta3 = 1.5
# p_occ = logistic(intercept
#                  + beta1 * sdm.scale(env_stack["Precipitation.Seasonality"])
#                  + beta2 * sdm.scale(env_stack["Min.Temperature.of.Coldest.Month"])
#                  + beta3 * sdm.scale(env_stack["Mean.Temperature.of.Wettest.Quarter"]))

intercept = -0.25
beta1 = -1
beta2 = 1
beta3 = 1.5
beta4 = 1
beta5 = -1
p_occ = logistic(intercept
                 + beta1 * sdm.scale(env_stack["Mean.Diurnal.Range"])
                 + beta2 * sdm.scale(env_stack["Isothermality"])
                 + beta3 * sdm.scale(env_stack["Mean.Temperature.of.Wettest.Quarter"])
                 + beta4 * sdm.scale(env_stack["Mean.Diurnal.Range"])**2   # quadratic term
                 + beta5 * sdm.scale(env_stack["Isothermality"])
                         * sdm.scale(env_stack["Mean.Temperature.of.Wettest.Quarter"]))   # interaction term

# get the values that correspond to the points we selected:
p_occs = sdm.extract(p_occ, env_stack, sim_pts["x"], sim_pts["y"])

# sample from these probabilities (a *different* seed than above; see Lab 1):
rng = np.random.default_rng(5678)
pres_abs = rng.binomial(n=1, p=p_occs)

# put response and variables together in a DataFrame:
modeling_df = sim_pts.copy()
modeling_df.insert(0, "presAbs", pres_abs)

# plot the result
absent = modeling_df["presAbs"] == 0     # True for absences, False for presences
ax = sdm.plot_map(p_occ, env_stack, title="Probability of occurrence")
ax.scatter(modeling_df.loc[absent, "x"], modeling_df.loc[absent, "y"], s=6, color="blue",
           label="non-detection")
ax.scatter(modeling_df.loc[~absent, "x"], modeling_df.loc[~absent, "y"], s=6, color="red",
           label="detection")    # ~ means "not"
ax.legend(loc="center left");

In [ ]:
# split into training and test (// is integer division: 1000 // 2 = 500)
train_data = modeling_df.iloc[:n_points // 2]   # rows 0-499
test_data = modeling_df.iloc[n_points // 2:]    # rows 500-999

(`sample_random()` returns the points in random order, so taking the first half for training and
the second half for testing is a random split.)

In R, the response must be a *factor* for randomForest to perform classification. In scikit-learn
you choose by picking the class instead: `RandomForestClassifier` for classification,
`RandomForestRegressor` for regression.

# Random Forests

## A preliminary note about the bootstrap

The papers we read referenced ~36% of the data as being out-of-bag (OOB) a few times. This is a
property of bootstrap sampling. Experiment with this code to develop your intuition.
[More explanation.](https://stats.stackexchange.com/questions/88980/why-on-average-does-each-bootstrap-sample-contain-roughly-two-thirds-of-observat?lq=1)

(Two Python notes: `.mean()` of True/False values is the proportion that are True, because True
counts as 1. And `for i in range(1000):` repeats the indented lines below it, with `i` = 0, 1, ...,
999.)

In [ ]:
rng = np.random.default_rng()   # no seed on purpose: run this cell a few times!

# Single Sample:
s = rng.choice(np.arange(1, 11), size=10, replace=True)   # bootstrap sample (with replacement)
print(s)
included = np.isin(np.arange(1, 11), s)   # was each datapoint included in the sample (at least once)?
print(included)
print(included.mean())   # proportion of original datapoints included in the bootstrap sample

# Repeat this 1000 times for a dataset with 100 samples and record the proportions:
props = np.empty(1000)
for i in range(1000):   # imagine that each iteration of this loop is drawing the subsample of data for one tree in a random forest
    boot = rng.choice(np.arange(1, 101), size=100, replace=True)
    props[i] = np.isin(np.arange(1, 101), boot).mean()

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(props)
ax.set_xlabel("props")
print(props.mean())   # pretty close to 63.2%? the other 36.8% are then OOB.

## Fit a random forest with default parameters

This RF is fit to all the variables, and it uses 500 trees and floor(sqrt(20)) = 4 variables tried
at each split (we have 20 predictors). It's fitting classification trees, so the outputs of each tree
are discrete (e.g., presence or absence). The final predictions are majority votes from the ensemble
of trees. R's randomForest prints a confusion matrix for the out-of-bag data points; below we compute
it ourselves.

We'll use all 20 environmental variables in every model in this lab. scikit-learn wants the
predictors (`X`) and the response (`y`) as separate objects, so let's set those up once:

In [ ]:
pred_vars = env_stack.names   # the 20 environmental variables
X_train = train_data[pred_vars]
y_train = train_data["presAbs"]
X_test = test_data[pred_vars]
y_test = test_data["presAbs"]
print(len(pred_vars), "predictors:", pred_vars)

Here is how R's randomForest settings map to scikit-learn's:

| randomForest (R) | scikit-learn | here |
|---|---|---|
| `ntree = 500` | `n_estimators` | scikit-learn's default is only 100, so we set 500 |
| `mtry = floor(sqrt(p))` | `max_features="sqrt"` | 4 of 20 (scikit-learn's default for classifiers too) |
| `nodesize = 1` (classification) | `min_samples_leaf=1` | trees are fully grown (the default in both) |
| OOB error is always computed | `oob_score=True` | you have to ask for it |
| `set.seed(1234)` | `random_state=1234` | makes the forest reproducible |

In [ ]:
model1 = RandomForestClassifier(n_estimators=500,      # number of trees (R: ntree)
                                max_features="sqrt",   # variables tried at each split (R: mtry)
                                oob_score=True,        # compute out-of-bag accuracy
                                random_state=1234)
model1.fit(X_train, y_train)

def print_oob(forest):
    """Print the OOB error rate and OOB confusion matrix, like print() of a randomForest in R."""
    # oob_decision_function_ holds each training point's OOB class probabilities (columns: class 0,
    # class 1); argmax(axis=1) picks the column with the larger one, i.e., the predicted class.
    oob_pred = forest.oob_decision_function_.argmax(axis=1)
    print("Number of trees:", forest.n_estimators,
          "| variables tried at each split:", forest.max_features)
    print("OOB estimate of error rate:", round(100 * (1 - forest.oob_score_), 1), "%")
    print(pd.crosstab(y_train.to_numpy(), oob_pred, rownames=["true"], colnames=["OOB predicted"]))

print_oob(model1)

We can plot how the out of bag error rate changes as more trees are grown. R's randomForest records
this as it goes (`model1$err.rate`); scikit-learn doesn't, but we can rebuild it with
`warm_start=True`. With that setting, calling `fit()` again after raising `n_estimators` *keeps* the
trees already grown and only adds the new ones. So we grow the forest 20 trees at a time and record
the OOB error (`1 - oob_score_`) after each step. (Each re-fit recomputes the OOB error from all the
trees grown so far.)

We'll do this again for other forests, so let's write a function. Because it uses the same
`random_state`, the warm-start forest grows exactly the same trees as `model1`, and the last point of
the curve equals `1 - model1.oob_score_`. (`_` is the conventional name for a returned value we
don't need.)

In [ ]:
def fit_with_oob_curve(tree_counts, max_features="sqrt"):
    """Grow a classification forest on the training data to each size in `tree_counts` in turn.

    Returns the fitted forest and a DataFrame with the OOB error rate at each size.
    """
    forest = RandomForestClassifier(n_estimators=tree_counts[0], max_features=max_features,
                                    oob_score=True, random_state=1234,
                                    warm_start=True,   # fit() adds trees instead of starting over
                                    n_jobs=-1)         # use all processor cores (see the end of the lab)
    curve = []
    for n in tree_counts:
        forest.set_params(n_estimators=n)   # ask for more trees...
        forest.fit(X_train, y_train)        # ...and grow only the new ones
        curve.append({"trees": n, "oob_error": 1 - forest.oob_score_})
    return forest, pd.DataFrame(curve)

_, curve1 = fit_with_oob_curve(tree_counts=range(20, 501, 20))   # 20, 40, ..., 500 trees
print("last point of the curve:", curve1["oob_error"].iloc[-1])
print("1 - model1.oob_score_:  ", 1 - model1.oob_score_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve1["trees"], curve1["oob_error"], "o-", markersize=3)
ax.set_xlabel("Number of trees fitted (iterations)")
ax.set_ylabel("OOB error rate");

We can use the predict function on the test data and evaluate. The default is that predictions from
a forest of classification trees are discrete (present or absent):

In [ ]:
preds1pa = model1.predict(X_test)
sdm.confusion_summary(y_test, preds1pa);

If we ask for `predict_proba()`, we can get numbers between 0 and 1 instead: one column per class, in
the order of `model1.classes_` (0, then 1), so column 1 is presence. These scores are amenable to
analysis with techniques like ROC curves as well.

There is one small difference from R here. randomForest's probability is the proportion of trees
that *vote* for presence. scikit-learn instead averages each tree's predicted probability (and
`predict()` picks the class with the larger average). But the leaves of a fully grown tree almost
always contain a single class, so each tree's "probability" is 0 or 1, i.e., a vote, and the two give
the same answer. Let's check by counting the votes ourselves. The list comprehension
`[... for tree in model1.estimators_]` repeats the prediction for each of the 500 trees.

In [ ]:
preds1prob = model1.predict_proba(X_test)[:, 1]   # column 1 = class 1 (presence)
print("classes:", model1.classes_)

# each tree's vote (0 or 1) for every test point (the trees inside a forest expect a plain array)
votes = np.array([tree.predict(X_test.to_numpy()) for tree in model1.estimators_])
vote_share = votes.mean(axis=0)   # proportion of trees voting "presence"
print("largest difference between vote share and predict_proba:",
      np.abs(vote_share - preds1prob).max())

In [ ]:
fpr1, tpr1, _ = roc_curve(y_test, preds1prob)
print("Area under the curve:", round(roc_auc_score(y_test, preds1prob), 4))

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr1, tpr1, color="black")
ax.plot([0, 1], [0, 1], color="grey", linestyle="--")
ax.set_xlabel("False positive rate (1 - specificity)")
ax.set_ylabel("True positive rate (sensitivity)");

(pROC in R draws the same curve with specificity on a reversed x-axis.)

We could also fit a random forest doing regression instead of classification, even though the
response only has two values. In scikit-learn we just use `RandomForestRegressor` with the same 0/1
response (unlike R, it gives no warning). This fits to minimize mean squared error between the 0/1
responses and the predictions, and the leaves of the trees contain numbers in [0,1] now.

Two of scikit-learn's regression defaults differ from randomForest's, so we set them to match R:

* `max_features`: randomForest tries floor(p/3) = floor(20/3) = 6 variables at each split of a
  regression tree. scikit-learn's default for regression is `max_features=1.0`, i.e., *all* 20
  variables, which turns the random forest into plain bagging (no random choice of variables).
* `min_samples_split`: randomForest stops splitting a regression-tree node once it holds 5 or fewer
  points (`nodesize = 5`). `min_samples_split=6` does the same in scikit-learn, whose default grows
  regression trees all the way down to single points.

In [ ]:
mtry_regr = len(pred_vars) // 3   # floor(20/3) = 6
model_regr = RandomForestRegressor(n_estimators=500,
                                   max_features=mtry_regr,   # R's default mtry for regression
                                   min_samples_split=6,      # R's default nodesize (5) for regression
                                   oob_score=True,
                                   random_state=1234)
model_regr.fit(X_train, y_train)

# oob_prediction_ holds each training point's OOB prediction; oob_score_ is the OOB R-squared
print("Mean of squared residuals (OOB):", round(np.mean((y_train - model_regr.oob_prediction_) ** 2), 4))
print("% Var explained (OOB):", round(100 * model_regr.oob_score_, 2))

We can compare the resulting models via AUC, among other options.

In [ ]:
preds_regr_prob = model_regr.predict(X_test)
print("Area under the curve:", round(roc_auc_score(y_test, preds_regr_prob), 4))
fpr_regr, tpr_regr, _ = roc_curve(y_test, preds_regr_prob)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr1, tpr1, color="black", linewidth=2, label="classification RF")
ax.plot(fpr_regr, tpr_regr, color="red", linewidth=2, label="regression RF")
ax.plot([0, 1], [0, 1], color="grey", linestyle="--")
ax.set_xlabel("False positive rate (1 - specificity)")
ax.set_ylabel("True positive rate (sensitivity)")
ax.legend(loc="lower right");

## More trees

Let's play with the parameters now. Here's a classification RF with 5000 trees instead of 500.

Since the warm-start forest ends up with exactly the trees a single fit would grow, from now on we
let our function fit the model and record the OOB curve at the same time. Recomputing the OOB error
from thousands of trees gets slow, so we record it less often as the forest grows. (Each 5000-tree
forest takes about half a minute.)

In [ ]:
tree_counts_5000 = [50, 100, 200, 300, 400, 500, 750, 1000, 1500, 2000, 3000, 4000, 5000]
model2, curve2 = fit_with_oob_curve(tree_counts_5000)
print_oob(model2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve2["trees"], curve2["oob_error"], "o-", markersize=3)
ax.set_xlabel("Number of trees fitted (iterations)")
ax.set_ylabel("OOB error rate");

In [ ]:
preds2pa = model2.predict(X_test)
sdm.confusion_summary(y_test, preds2pa);

In [ ]:
preds2prob = model2.predict_proba(X_test)[:, 1]
print("Area under the curve:", round(roc_auc_score(y_test, preds2prob), 4))
fpr2, tpr2, _ = roc_curve(y_test, preds2prob)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr2, tpr2, color="black")
ax.plot([0, 1], [0, 1], color="grey", linestyle="--")
ax.set_xlabel("False positive rate (1 - specificity)")
ax.set_ylabel("True positive rate (sensitivity)");

The results were very similar for 500 and 5000 trees. Here is the proportion of test points where
the two forests predict the same class:

In [ ]:
np.mean(preds1pa == preds2pa)

## Varying mtry

The mtry parameter (`max_features` in scikit-learn) influences the probability that a truly relevant
variable will be in the subset considered at each split. If there are a small proportion of relevant
variables, a higher mtry may be warranted. Let's try halving and doubling the default:

In [ ]:
model3, curve3 = fit_with_oob_curve(tree_counts_5000, max_features=2)
print_oob(model3)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve3["trees"], curve3["oob_error"], "o-", markersize=3)
ax.set_xlabel("Number of trees fitted (iterations)")
ax.set_ylabel("OOB error rate")
plt.show()

preds3pa = model3.predict(X_test)
sdm.confusion_summary(y_test, preds3pa)
print("\nagreement with model2:", np.mean(preds2pa == preds3pa))

In [ ]:
model4, curve4 = fit_with_oob_curve(tree_counts_5000, max_features=8)
print_oob(model4)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve4["trees"], curve4["oob_error"], "o-", markersize=3)
ax.set_xlabel("Number of trees fitted (iterations)")
ax.set_ylabel("OOB error rate")
plt.show()

preds4pa = model4.predict(X_test)
sdm.confusion_summary(y_test, preds4pa)
print("\nagreement with model2:", np.mean(preds2pa == preds4pa))

Very little difference!

## Variable importances

One of the papers below referenced four variable importance measures; two are implemented in the
randomForest package. The other two are described in Breiman's 2002 manual cited below and deal with
the margin of the cases, which is the difference between the proportion of votes for its true class
and the proportion of votes for the next best class (i.e., the amount the algorithm is getting this
case correct by). These are not in R's randomForest or in scikit-learn.

scikit-learn has versions of the two measures in randomForest, under different names:

* **Impurity-based importance** (`feature_importances_`) ≈ R's **MeanDecreaseGini**: the
  total decrease in Gini impurity from all splits on a variable, averaged over the trees.
  scikit-learn rescales these to sum to 1, so only their *relative* sizes are comparable with R's.
  They come from the training data during fitting, and they tend to favor variables with many
  possible split points.
* **Permutation importance** (`permutation_importance()`) ≈ R's **MeanDecreaseAccuracy**: shuffle one
  variable's values, so that it no longer carries any information, and measure how much the accuracy
  drops. R does this inside each tree, using that tree's out-of-bag points. scikit-learn's function
  shuffles a column of whatever data you give it and measures the drop in the whole forest's score.
  We give it the test data, which the forest has never seen, and shuffle each variable 10 times.
  Expect smaller and noisier values than R's: when one variable is shuffled, the forest as a whole
  can lean on trees that split on correlated variables, so its accuracy drops less than a single
  tree's does.

The R version uses the 5000-tree forest here. Each permutation means predicting with every tree, so
with 5000 trees this takes several minutes in scikit-learn; we use the 500-tree `model1` instead (the
two forests make almost identical predictions, as we saw). Here they are as a table:

In [ ]:
perm1 = permutation_importance(model1, X_test, y_test, scoring="accuracy",
                               n_repeats=10, random_state=1234)
importance1 = pd.DataFrame({"MeanDecreaseAccuracy (permutation)": perm1.importances_mean,
                            "MeanDecreaseGini (impurity)": model1.feature_importances_},
                           index=pred_vars)
importance1.round(4)

And as plots, like R's `varImpPlot()`:

In [ ]:
def plot_importances(importance_table):
    """Dot chart of each column of an importance table, sorted, like varImpPlot() in R."""
    n_columns = importance_table.shape[1]
    fig, axes = plt.subplots(1, n_columns, figsize=(12, 6), constrained_layout=True)
    for ax, column in zip(axes, importance_table.columns):
        values = importance_table[column].sort_values()
        ax.plot(values.to_numpy(), range(len(values)), "o")
        ax.set_yticks(range(len(values)), values.index, fontsize=8)
        ax.set_title(column, fontsize=10)
        ax.grid(axis="y", linestyle=":")
    return fig

plot_importances(importance1);

We can also make partial dependence plots to examine the relationship learned by the model for a
variable (e.g., Mean Diurnal Range) and the species response. `PartialDependenceDisplay` does the same
averaging as R's `partialPlot()`. For a classifier it shows the predicted probability of the positive
class (here presence, class 1) directly, while R's `partialPlot()` shows it on a logit scale, so the
curves have similar shapes but different y-axes. We use 51 grid points over the full range of the
variable, as R does (scikit-learn's default spans only the 5th to 95th percentiles). Again we use
`model1` to save time.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
PartialDependenceDisplay.from_estimator(model1, X_train, ["Mean.Diurnal.Range"],
                                        grid_resolution=51, percentiles=(0, 1), ax=ax);

## Visualize a proximity matrix

Valavi et al (2021) discuss class overlap as an issue that affects performance in RFs. Proximity
matrices are one way to visualize this. The proximity matrix reports the proportion of times each
pair of data points falls into the same terminal node in a tree of the random forest. (By default,
R's randomForest counts only the trees for which *both* points were out-of-bag, so the proximities
are not inflated by trees that were trained on those points. We'll do the same.) So, it is a
similarity matrix with dimension nPoints x nPoints (here, the 500 training points).

scikit-learn doesn't compute proximities, but we can, from two parts of the fitted forest:

* `model2.apply(X_train)` gives, for every point and every tree, the leaf (terminal node) the point
  lands in;
* `model2.estimators_samples_` lists, for every tree, the points in its bootstrap sample (the rest are
  out-of-bag).

The line `leaves[:, t][:, None] == leaves[:, t][None, :]` compares every point's leaf with every
other point's leaf at once, giving a 500 x 500 table of True/False (numpy calls this
"broadcasting"). This cell takes a few seconds.

In [ ]:
leaves = model2.apply(X_train)        # shape (500 points, 5000 trees): leaf of each point in each tree
inbag = model2.estimators_samples_    # one array of in-bag point numbers per tree
n_train, n_trees = leaves.shape

same_leaf = np.zeros((n_train, n_train))   # trees where points i and j are both OOB and share a leaf
both_oob = np.zeros((n_train, n_train))    # trees where points i and j are both OOB
for t in range(n_trees):
    oob = np.ones(n_train, dtype=bool)
    oob[inbag[t]] = False                  # in-bag points are not out-of-bag
    pair_oob = np.outer(oob, oob)          # True where both points are OOB in tree t
    pair_same = leaves[:, t][:, None] == leaves[:, t][None, :]   # True where both share a leaf
    both_oob += pair_oob
    same_leaf += pair_oob & pair_same

proximity = same_leaf / both_oob
np.fill_diagonal(proximity, 1)
np.round(proximity[:5, :5], 3)   # the first 5 x 5 = 25 pairs

Note that the diagonal is 1: each point always falls into the same terminal node as itself.

We can look at a 2-dimensional representation of the distance matrix given by (1 - proximity), where
the colors are the two classes (presence and absence). R's `MDSplot()` uses *classical*
multidimensional scaling (`cmdscale()`), which finds coordinates whose straight-line distances match
the given distances as closely as possible. It only takes a few lines of numpy (`@` is matrix
multiplication). (scikit-learn's `MDS` class uses a different, iterative method, and its arguments
are changing between versions, so we do it by hand.)

In [ ]:
# classical multidimensional scaling, as in R's cmdscale()
dist = 1 - proximity
centering = np.eye(n_train) - np.ones((n_train, n_train)) / n_train
B = -0.5 * centering @ dist**2 @ centering        # "double-centred" squared distances
eigenvalues, eigenvectors = np.linalg.eigh(B)     # eigh returns them in increasing order
top2 = np.argsort(eigenvalues)[::-1][:2]          # positions of the two largest
mds_points = eigenvectors[:, top2] * np.sqrt(eigenvalues[top2])

present = y_train.to_numpy() == 1
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(mds_points[~present, 0], mds_points[~present, 1], s=10, color="blue", label="absence")
ax.scatter(mds_points[present, 0], mds_points[present, 1], s=10, color="red", label="presence")
ax.set_xlabel("Dim 1")
ax.set_ylabel("Dim 2")
ax.legend();

## Visualize the predictive model

If we give the full stack of rasters to the random forest, we can see the learned occurrence
surface. Column 1 of `predict_proba()` is the probability of presence. (This takes up to a minute:
5000 trees, applied to about 109,000 land cells.)

In [ ]:
grid = env_stack.to_frame()   # one row per land cell
p = env_stack.to_map(model2.predict_proba(grid[pred_vars])[:, 1], index=grid.index)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
sdm.plot_map(p_occ, env_stack, ax=axes[0], title="True Prob. of Occurrence", vmin=0, vmax=1)
sdm.plot_map(p, env_stack, ax=axes[1], title="RF prediction", vmin=0, vmax=1);

## Abundance modeling

Random forests can also work on real-valued (e.g., abundance) data. Let's simulate some and try it
out.

In [ ]:
intercept = 2
beta1 = 0.5
beta2 = -0.5
beta3 = 0.25
beta4 = -0.05
beta5 = -0.2
lambda_map = np.exp(intercept
                    + beta1 * sdm.scale(env_stack["Mean.Diurnal.Range"])
                    + beta2 * sdm.scale(env_stack["Isothermality"])
                    + beta3 * sdm.scale(env_stack["Mean.Temperature.of.Wettest.Quarter"])
                    + beta4 * sdm.scale(env_stack["Mean.Diurnal.Range"])**2   # quadratic term
                    + beta5 * sdm.scale(env_stack["Isothermality"])
                            * sdm.scale(env_stack["Mean.Temperature.of.Wettest.Quarter"]))   # interaction term

sdm.plot_map(lambda_map, env_stack, title="Mean abundance (lambda)");

In [ ]:
# get the mean abundances that correspond to the points we selected:
lambdas = sdm.extract(lambda_map, env_stack, sim_pts["x"], sim_pts["y"])

# sample from these means (again with a new seed, different from the ones used above):
rng = np.random.default_rng(2468)
abund = rng.poisson(lam=lambdas)   # sampling from a Poisson instead of binomial now
train_abund = abund[:n_points // 2]
test_abund = abund[n_points // 2:]
# put response and variables together in a DataFrame:
abund_df = train_data[pred_vars].copy()
abund_df.insert(0, "abund", train_abund)

# plot the three relevant climate variables and the mean abundance surface
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), constrained_layout=True)
for ax, name in zip(axes.flat, ["Mean.Diurnal.Range", "Isothermality",
                                "Mean.Temperature.of.Wettest.Quarter"]):
    sdm.plot_map(env_stack[name], env_stack, ax=ax, title=name)
sdm.plot_map(lambda_map, env_stack, ax=axes[1, 1], title="Mean Abundance");

Note that these data are generated from a Poisson distribution, but the random forest is not
considering this. In some sense, it's treating them as Gaussian, since it's measuring error with MSE.
For regression, randomForest's default mtry=floor(20/3)=6; as above, we set `max_features` to match,
since scikit-learn's default would use all 20 variables.

We time how long it takes to train this RF with `time.perf_counter()` (a stopwatch: we read it before
and after, and subtract), because in a moment we'll compare it against a faster setup. This forest
uses one processor core, like R's randomForest. (`f"...{x:.2f}..."` is an *f-string*: it puts the
value of `x` into the text, rounded to 2 decimals.)

In [ ]:
X_abund = abund_df[pred_vars]
y_abund = abund_df["abund"]

start = time.perf_counter()
model5 = RandomForestRegressor(n_estimators=1000,
                               max_features=mtry_regr,   # R's default mtry for regression
                               min_samples_split=6,      # R's default nodesize (5) for regression
                               oob_score=True,
                               random_state=1234)        # n_jobs not set: one core
model5.fit(X_abund, y_abund)
time_one_core = time.perf_counter() - start
print(f"one core: {time_one_core:.2f} sec elapsed")

print("Mean of squared residuals (OOB):", round(np.mean((y_abund - model5.oob_prediction_) ** 2), 3))
print("% Var explained (OOB):", round(100 * model5.oob_score_, 2))

Now, OOB estimates are of MSE instead of misclassification rate. As before, we grow a copy of the
forest with `warm_start=True` (20 trees at a time) to see how the OOB MSE changes as trees are added.
For a regression forest, `oob_score_` is an R-squared, so we compute the MSE from the OOB
predictions.

In [ ]:
forest = RandomForestRegressor(n_estimators=20, max_features=mtry_regr, min_samples_split=6,
                               oob_score=True, random_state=1234, warm_start=True, n_jobs=-1)
curve5 = []
for n in range(20, 1001, 20):
    forest.set_params(n_estimators=n)
    forest.fit(X_abund, y_abund)
    curve5.append({"trees": n, "oob_mse": np.mean((y_abund - forest.oob_prediction_) ** 2)})
curve5 = pd.DataFrame(curve5)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve5["trees"], curve5["oob_mse"], "o", markersize=3)
ax.set_xlabel("Number of trees fitted (iterations)")
ax.set_ylabel("OOB MSE");

We can compare against the MSE on predictions for the independent test set.

In [ ]:
preds5 = model5.predict(X_test)
np.mean((preds5 - test_abund) ** 2)

We can look at variable importance with similar measures. For regression, the impurity is the
squared error, so impurity-based importance ≈ R's **IncNodePurity**, and permutation importance
measured as the increase in test MSE (`scoring="neg_mean_squared_error"`) ≈ R's **%IncMSE**.

In [ ]:
perm5 = permutation_importance(model5, X_test, test_abund, scoring="neg_mean_squared_error",
                               n_repeats=5, random_state=1234)
importance5 = pd.DataFrame({"%IncMSE (permutation)": perm5.importances_mean,
                            "IncNodePurity (impurity)": model5.feature_importances_},
                           index=pred_vars)
print(importance5.round(4))
plot_importances(importance5);

## The ranger package

Valavi et al (2021) suggest the R package 'ranger' has a fast implementation of RFs, whereas the R
version of this lab used the 'randomForest' package above. Python doesn't have this split:
scikit-learn's forests are already written in fast compiled code. Part of ranger's speed comes from
using all of your computer's processor cores at once, which scikit-learn also does when you set
`n_jobs=-1` (the default uses one core). Let's time the same forest that way:

In [ ]:
start = time.perf_counter()
model5_parallel = RandomForestRegressor(n_estimators=1000, max_features=mtry_regr,
                                        min_samples_split=6, oob_score=True, random_state=1234,
                                        n_jobs=-1)   # use all processor cores
model5_parallel.fit(X_abund, y_abund)
time_all_cores = time.perf_counter() - start
print(f"all {os.cpu_count()} cores: {time_all_cores:.2f} sec elapsed (one core: {time_one_core:.2f} sec)")

How much faster it is depends on how many cores your computer has (printed above).

Neither ranger nor scikit-learn records the OOB error as trees are added, which is why we used the
`warm_start` loop above.

The test MSE is identical to the one-core forest: `n_jobs` only changes how the work is shared among
the cores, not the trees themselves (both use `random_state=1234`). (In R, randomForest and ranger are
different implementations with different defaults, so their results are similar but not identical.)

In [ ]:
preds_parallel = model5_parallel.predict(X_test)
print("test MSE, all cores:", np.mean((preds_parallel - test_abund) ** 2))
print("test MSE, one core: ", np.mean((preds5 - test_abund) ** 2))

And the importance values are the same, too:

In [ ]:
np.allclose(model5_parallel.feature_importances_, model5.feature_importances_)

ranger has no partial dependence plots built in, so the R version uses the separate 'pdp' package.
scikit-learn's `PartialDependenceDisplay` works with any fitted model; here it is for the abundance
forest, averaging the predictions over the training data (`method="brute"`), as pdp does:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
PartialDependenceDisplay.from_estimator(model5_parallel, X_abund, ["Mean.Diurnal.Range"],
                                        grid_resolution=51, percentiles=(0, 1), method="brute", ax=ax);

> **Exercise:** Choose any aspect of RFs that you are curious about and explore it further. Report
> briefly on your motivation, design of your experiment, results, and conclusions. To play with
> changing the sampling of the classes: in R you would manipulate the sample.fraction parameter in
> ranger or the sampsize parameter in randomForest. In scikit-learn, `max_samples` sets how many
> points go into each tree's bootstrap sample, and `class_weight="balanced_subsample"` reweights the
> classes within each tree's bootstrap sample. scikit-learn has no built-in option for drawing a fixed
> number of points from *each* class, as randomForest's sampsize does.

# Turn in

**(a Canvas discussion post, by the due date on Canvas)** Please post to the online discussion of
this lab. Please include brief descriptions of your exploration of the exercise, with any relevant
plots. Please also include, if applicable:

* Any key take-aways from this lab that you want to remember for any future projects you may conduct
  in which you use species distribution models.
* Any remaining questions or confusions do you have that you might like to learn more about.

# Further Resources

Evans, J. S., Murphy, M. A., Holden, Z. A., & Cushman, S. A. (2010). Modeling Species Distribution
and Change Using Random Forest. In Predictive Species and Habitat Modeling in Landscape Ecology:
Concepts and Applications (pp. 139–159). [doi](https://doi.org/10.1007/978-1-4419-7390-0_8)

Cutler, D. R., Edwards, T. C., Jr., Beard, K. H., Cutler, A., Hess, K. T., Gibson, J., & Lawler, J.
J. (2007). Random Forests for Classification in Ecology. Ecology, 88(11), 2783-2792.
[doi](https://doi.org/10.1890/07-0539.1)

Prasad, A. M., Iverson, L. R., & Liaw, A. (2006). Newer classification and regression tree
techniques: Bagging and random forests for ecological prediction. Ecosystems, 9(2), 181-199.

Liaw, A., & Wiener, M. (2002). Classification and Regression by randomForest. R News, 2(3), 18-22.

Breiman, L. [Manual on setting up, using, and understanding random forests
v3.1](https://www.stat.berkeley.edu/~breiman/Using_random_forests_V3.1.pdf), 2002.

Valavi, R., Elith, J., Lahoz-Monfort, J. J., & Guillera-Arroita, G. (2021). Modelling species
presence-only data with random forests. Ecography, 44, 1731-1742.
[doi](https://doi.org/10.1111/ecog.05615)

scikit-learn user guide: [Random forests](https://scikit-learn.org/stable/modules/ensemble.html#random-forests)
and [permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html).

Technical statistical foundations:
Breiman, L. (2001). Random forests. Machine Learning, 45(1), 5-32.
[doi](https://doi.org/10.1023/A:1010933404324)